# HKU-FYP Master Pipeline (v1)

Goal: deliver a reproducible **first version this weekend** for binary direction prediction using Yahoo Finance data (no GPU).


## Scope (v1)
- Universe: pilot 20 tickers (S&P100 subset)
- Horizons: 1D / 3D / 5D / 10D
- Features: technical only first (extend to fundamental+macro later)
- Models: Logistic Regression, Random Forest, XGBoost (fallback to GradientBoosting if xgboost unavailable)
- Metrics: Accuracy, Balanced Accuracy, F1


In [ ]:
# If needed, install dependencies in notebook environment
# !pip install yfinance pandas numpy scikit-learn matplotlib seaborn xgboost


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


In [ ]:
# Pilot tickers (editable)
TICKERS = [
    'AAPL','MSFT','NVDA','AMZN','GOOGL','META','TSLA','JPM','V','MA',
    'UNH','HD','PG','JNJ','XOM','AVGO','COST','KO','PEP','MRK'
]

START_DATE = '2005-01-01'
END_DATE = None
HORIZONS = [1,3,5,10]


In [ ]:
def download_ohlcv(tickers, start=START_DATE, end=END_DATE):
    frames = []
    for t in tickers:
        df = yf.download(t, start=start, end=end, auto_adjust=True, progress=False)
        if df is None or df.empty:
            continue
        df = df[['Open','High','Low','Close','Volume']].copy()
        df.columns = ['open','high','low','close','volume']
        df['ticker'] = t
        df['date'] = df.index
        frames.append(df.reset_index(drop=True))
    if not frames:
        raise ValueError('No data downloaded.')
    return pd.concat(frames, ignore_index=True)


In [ ]:
def add_technical_features(df):
    out = []
    for t, g in df.groupby('ticker'):
        g = g.sort_values('date').copy()
        g['ret_1d'] = g['close'].pct_change(1)
        g['ret_5d'] = g['close'].pct_change(5)
        g['ret_10d'] = g['close'].pct_change(10)
        g['vol_10d'] = g['ret_1d'].rolling(10).std()
        g['vol_20d'] = g['ret_1d'].rolling(20).std()
        g['ma_5'] = g['close'].rolling(5).mean()
        g['ma_20'] = g['close'].rolling(20).mean()
        g['ma_ratio_5_20'] = g['ma_5'] / g['ma_20']
        g['mom_10'] = g['close'] / g['close'].shift(10) - 1
        out.append(g)
    return pd.concat(out, ignore_index=True)


In [ ]:
def add_labels(df, horizons=HORIZONS):
    out = []
    for t, g in df.groupby('ticker'):
        g = g.sort_values('date').copy()
        for h in horizons:
            fwd_ret = g['close'].shift(-h) / g['close'] - 1
            g[f'label_{h}d'] = (fwd_ret > 0).astype(int)
        out.append(g)
    return pd.concat(out, ignore_index=True)


In [ ]:
FEATURES = ['ret_1d','ret_5d','ret_10d','vol_10d','vol_20d','ma_ratio_5_20','mom_10']

def time_split(df, train_ratio=0.7, val_ratio=0.15):
    dates = np.sort(df['date'].unique())
    n = len(dates)
    d1 = dates[int(n*train_ratio)]
    d2 = dates[int(n*(train_ratio+val_ratio))]
    train = df[df['date'] < d1]
    val = df[(df['date'] >= d1) & (df['date'] < d2)]
    test = df[df['date'] >= d2]
    return train, val, test


In [ ]:
def get_models(random_state=42):
    models = {
        'logreg': LogisticRegression(max_iter=1000, n_jobs=None),
        'rf': RandomForestClassifier(n_estimators=200, max_depth=None, random_state=random_state, n_jobs=-1),
    }
    try:
        from xgboost import XGBClassifier
        models['xgb'] = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=random_state,
            n_jobs=-1
        )
    except Exception:
        models['gb_fallback'] = GradientBoostingClassifier(random_state=random_state)
    return models


In [ ]:
def evaluate_models(df, horizon=1):
    label_col = f'label_{horizon}d'
    use = df[['date','ticker'] + FEATURES + [label_col]].dropna().copy()
    train, val, test = time_split(use)

    X_train, y_train = train[FEATURES], train[label_col]
    X_test, y_test = test[FEATURES], test[label_col]

    rows = []
    for name, model in get_models().items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rows.append({
            'horizon': f'{horizon}D',
            'model': name,
            'accuracy': accuracy_score(y_test, pred),
            'balanced_accuracy': balanced_accuracy_score(y_test, pred),
            'f1': f1_score(y_test, pred, zero_division=0),
            'test_samples': len(y_test)
        })
    return pd.DataFrame(rows)


In [ ]:
raw = download_ohlcv(TICKERS)
feat = add_technical_features(raw)
all_df = add_labels(feat)

results = []
for h in HORIZONS:
    results.append(evaluate_models(all_df, horizon=h))
results_df = pd.concat(results, ignore_index=True).sort_values(['horizon','accuracy'], ascending=[True, False])
results_df


In [ ]:
# Save first-version outputs
import os
os.makedirs('../outputs/metrics', exist_ok=True)
results_df.to_csv('../outputs/metrics/pilot_v1_metrics.csv', index=False)
print('Saved: ../outputs/metrics/pilot_v1_metrics.csv')


## Next Steps
1. Add fundamental + macro features
2. Expand from pilot 20 names to broader S&P coverage
3. Add confusion matrix / per-horizon plots for report figures
